In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 192) / Test: (90067, 191)


In [2]:
# Feature_refinement 재적용 + 피처 준비
# 중요도 0 제거
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

# 교호작용 추가
for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)

print('피처 정제 완료 / Train:', train.shape)

피처 정제 완료 / Train: (256351, 199)


In [3]:
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

# 범주형 컬럼 (StringDtype 포함)
cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

# CatBoost용 인덱스
cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

# LightGBM용 category dtype
X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

피처: 197개 / 범주형: 21개
scale_pos_weight: 2.87


In [4]:
# CatBoost 5-fold (Optuna 최적 파라미터)
N_SPLITS = 5
SEED     = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

cat_params = {
    'iterations':            794,
    'learning_rate':         0.030094391673729362,
    'depth':                 7,
    'l2_leaf_reg':           7.433949857398995,
    'bagging_temperature':   0.3980358270898108,
    'random_strength':       1.3075975210328405,
    'border_count':          108,
    'loss_function':         'Logloss',
    'eval_metric':           'AUC',
    'scale_pos_weight':      spw,
    'random_seed':           SEED,
    'early_stopping_rounds': 50,
    'verbose':               False,
    'use_best_model':        True,
}

oof_cat  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))
cat_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr  = X.iloc[tr_idx].reset_index(drop=True)
    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_tr  = y.iloc[tr_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    train_pool = Pool(X_tr, y_tr, cat_features=cat_feature_indices)
    val_pool   = Pool(X_val, y_val, cat_features=cat_feature_indices)
    test_pool  = Pool(X_test, cat_features=cat_feature_indices)

    model = CatBoostClassifier(**cat_params)
    model.fit(train_pool, eval_set=val_pool)

    val_pred = model.predict_proba(val_pool)[:, 1]
    fold_auc = roc_auc_score(y_val, val_pred)

    oof_cat[val_idx]  = val_pred
    test_cat         += model.predict_proba(test_pool)[:, 1] / N_SPLITS
    cat_fold_scores.append(fold_auc)
    print(f'[CatBoost] Fold {fold+1} | AUC: {fold_auc:.5f}')

cat_auc = roc_auc_score(y, oof_cat)
print(f'\n✅ CatBoost OOF AUC: {cat_auc:.5f} ± {np.std(cat_fold_scores):.5f}')

[CatBoost] Fold 1 | AUC: 0.73848
[CatBoost] Fold 2 | AUC: 0.74320
[CatBoost] Fold 3 | AUC: 0.74083
[CatBoost] Fold 4 | AUC: 0.73832
[CatBoost] Fold 5 | AUC: 0.74077

✅ CatBoost OOF AUC: 0.74031 ± 0.00180


In [5]:
# LightGBM 5-fold
lgb_params = {
    'objective':        'binary',
    'metric':           'auc',
    'boosting_type':    'gbdt',
    'num_leaves':       63,
    'max_depth':        7,
    'learning_rate':    0.03,
    'n_estimators':     1000,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'scale_pos_weight': spw,
    'random_state':     SEED,
    'verbose':         -1,
}

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
lgb_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgb, y)):
    X_tr  = X_lgb.iloc[tr_idx]
    X_val = X_lgb.iloc[val_idx]
    y_tr  = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(-1)
        ]
    )

    val_pred = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, val_pred)

    oof_lgb[val_idx]  = val_pred
    test_lgb         += model.predict_proba(X_test_lgb)[:, 1] / N_SPLITS
    lgb_fold_scores.append(fold_auc)
    print(f'[LightGBM] Fold {fold+1} | AUC: {fold_auc:.5f} | Best iter: {model.best_iteration_}')

lgb_auc = roc_auc_score(y, oof_lgb)
print(f'\n✅ LightGBM OOF AUC: {lgb_auc:.5f} ± {np.std(lgb_fold_scores):.5f}')

[LightGBM] Fold 1 | AUC: 0.73751 | Best iter: 190
[LightGBM] Fold 2 | AUC: 0.74167 | Best iter: 218
[LightGBM] Fold 3 | AUC: 0.73995 | Best iter: 225
[LightGBM] Fold 4 | AUC: 0.73761 | Best iter: 159
[LightGBM] Fold 5 | AUC: 0.73962 | Best iter: 190

✅ LightGBM OOF AUC: 0.73926 ± 0.00156


In [6]:
# 최적 가중치 탐색 + 소프트 보팅
# 가중치 그리드 탐색 (0.0~1.0, 0.05 단위)
best_w, best_auc = 0.5, 0.0
results = []

for w_cat in np.arange(0.0, 1.05, 0.05):
    w_lgb = 1 - w_cat
    blended = oof_cat * w_cat + oof_lgb * w_lgb
    auc = roc_auc_score(y, blended)
    results.append((round(w_cat, 2), round(w_lgb, 2), auc))
    if auc > best_auc:
        best_auc = auc
        best_w   = w_cat

results_df = pd.DataFrame(results, columns=['w_catboost','w_lgbm','oof_auc'])
print('=== 가중치별 OOF AUC ===')
display(results_df.sort_values('oof_auc', ascending=False).head(10))

print(f'\n✅ CatBoost 단독  OOF AUC: {cat_auc:.5f}')
print(f'✅ LightGBM 단독  OOF AUC: {lgb_auc:.5f}')
print(f'✅ 앙상블 최적    OOF AUC: {best_auc:.5f}  (CatBoost {best_w:.2f} : LightGBM {1-best_w:.2f})')
print(f'베이스라인 대비: {best_auc - 0.74008:+.5f}')

=== 가중치별 OOF AUC ===


,w_catboost,w_lgbm,oof_auc
17,0.85,0.15,0.740353
16,0.80,0.20,0.740351
18,0.90,0.10,0.740348
15,0.75,0.25,0.740342
19,0.95,0.05,0.740335
14,0.70,0.30,0.740324
20,1.00,0.00,0.740314
13,0.65,0.35,0.740299
12,0.60,0.40,0.740266
11,0.55,0.45,0.740226



✅ CatBoost 단독  OOF AUC: 0.74031
✅ LightGBM 단독  OOF AUC: 0.73926
✅ 앙상블 최적    OOF AUC: 0.74035  (CatBoost 0.85 : LightGBM 0.15)
베이스라인 대비: +0.00027


In [8]:
# 제출 파일 생성
# 최적 가중치로 test 예측 블렌딩
test_blended = test_cat * best_w + test_lgb * (1 - best_w)

# sample_submission 컬럼 구조 그대로 사용
sample_sub = pd.read_csv('/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw/sample_submission.csv') 
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_blended

submission.to_csv('submission_ensemble.csv', index=False, encoding='utf-8-sig')

print('저장 완료: submission_ensemble.csv')
print(f'컬럼: {submission.columns.tolist()}')
print(f'\n최종 OOF AUC 요약')
print(f'  CatBoost  : {cat_auc:.5f}')
print(f'  LightGBM  : {lgb_auc:.5f}')
print(f'  앙상블     : {best_auc:.5f}  (w_cat={best_w:.2f}, w_lgb={1-best_w:.2f})')
print(f'  베이스라인 : 0.74008')
print(f'  개선폭     : {best_auc - 0.74008:+.5f}')
display(submission.head())

저장 완료: submission_ensemble.csv
컬럼: ['ID', 'probability']

최종 OOF AUC 요약
  CatBoost  : 0.74031
  LightGBM  : 0.73926
  앙상블     : 0.74035  (w_cat=0.85, w_lgb=0.15)
  베이스라인 : 0.74008
  개선폭     : +0.00027


,ID,probability
0,TEST_00000,0.002847
1,TEST_00001,0.018273
2,TEST_00002,0.341198
3,TEST_00003,0.243556
4,TEST_00004,0.729969
